# Notebook 05: Quantum Teleportation

---

**Author:** Milan Amrut Joshi  
**Date:** 2026-03-25  
**Prerequisites:** Notebooks 01-04 (especially Bell states and entanglement)  
**Framework:** Qiskit 1.x

---

## Table of Contents

1. What is Quantum Teleportation?
2. The No-Cloning Theorem
3. The Teleportation Protocol
4. Full Mathematical Derivation
5. Step-by-Step Circuit Construction
6. Implementation with Qiskit
7. Verification: Teleporting Known States
8. Teleporting Arbitrary States
9. Classical vs Quantum Communication
10. Exercises

## 1. What is Quantum Teleportation?

**Quantum teleportation** is a protocol that transfers a quantum state from one location (Alice) to another (Bob) using:
1. A shared **entangled pair** (Bell state)
2. **Two classical bits** of communication

### Key Points

- The original state is **destroyed** at Alice's end (no cloning!)
- No physical matter or energy is "teleported" — only quantum information
- It does **NOT** allow faster-than-light communication (requires classical channel)
- It was first proposed by Bennett et al. in 1993 and first demonstrated experimentally in 1997

### Overview Diagram

```
        Alice                                    Bob
    ┌──────────┐                            ┌──────────┐
    │          │                            │          │
    │  |ψ⟩     │   2 classical bits         │   |ψ⟩    │
    │  (input) │ ─────────────────────────> │  (output)│
    │          │                            │          │
    │  qubit A │~~~~entanglement~~~~────────│  qubit B │
    │  (Bell)  │    (shared EPR pair)       │  (Bell)  │
    │          │                            │          │
    └──────────┘                            └──────────┘
    
    Step 1: Alice & Bob share entangled pair
    Step 2: Alice performs Bell measurement on her qubits
    Step 3: Alice sends 2 classical bits to Bob
    Step 4: Bob applies correction based on Alice's bits
    Step 5: Bob has |ψ⟩!
```

### Why is This Remarkable?

1. Alice teleports $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ without knowing $\alpha$ and $\beta$
2. Only 2 classical bits are sent, but the state contains continuous (infinite) information
3. This is possible because the entangled pair was set up in advance

## 2. The No-Cloning Theorem

Before diving into teleportation, we must understand why it's needed: you **cannot copy** a quantum state.

### Theorem (Wootters, Zurek, Dieks, 1982)

There is no unitary operation $U$ that can clone an arbitrary quantum state:

$$\nexists \; U : U|\psi\rangle|0\rangle = |\psi\rangle|\psi\rangle \quad \forall \; |\psi\rangle$$

### Proof by Contradiction

Suppose such a $U$ exists. Then for two states $|\psi\rangle$ and $|\phi\rangle$:

$$U|\psi\rangle|0\rangle = |\psi\rangle|\psi\rangle \quad \text{...(1)}$$
$$U|\phi\rangle|0\rangle = |\phi\rangle|\phi\rangle \quad \text{...(2)}$$

Taking the inner product of (1) and (2):

$$\langle\psi|\phi\rangle \cdot \langle 0|0\rangle = \langle\psi|\phi\rangle \cdot \langle\psi|\phi\rangle$$

Since $\langle 0|0\rangle = 1$ and $U$ is unitary (preserves inner products):

$$\langle\psi|\phi\rangle = (\langle\psi|\phi\rangle)^2$$

This means $\langle\psi|\phi\rangle (1 - \langle\psi|\phi\rangle) = 0$.

So either $\langle\psi|\phi\rangle = 0$ (orthogonal) or $\langle\psi|\phi\rangle = 1$ (identical).

**Conclusion:** $U$ can only clone states that are either identical or orthogonal — it cannot clone arbitrary states. $\blacksquare$

### Implications

| What you CAN do | What you CANNOT do |
|-----------------|--------------------|
| Clone known basis states | Clone an unknown state |
| Create copies of $|0\rangle$ or $|1\rangle$ | Copy $\alpha|0\rangle + \beta|1\rangle$ |
| Move a state (teleportation) | Duplicate a state |

This is why teleportation **destroys** the original — it moves the state, it doesn't copy it.

In [ ]:
# Demonstrate the no-cloning theorem
import numpy as np

# Try to construct a "cloning" unitary for |0⟩ and |1⟩
# U|0⟩|0⟩ = |0⟩|0⟩ (clone |0⟩)  →  maps |00⟩ to |00⟩
# U|1⟩|0⟩ = |1⟩|1⟩ (clone |1⟩)  →  maps |10⟩ to |11⟩

# This works for the basis states: it's just CNOT!
CNOT = np.array([
    [1, 0, 0, 0],  # |00⟩ → |00⟩
    [0, 1, 0, 0],  # |01⟩ → |01⟩
    [0, 0, 0, 1],  # |10⟩ → |11⟩
    [0, 0, 1, 0],  # |11⟩ → |10⟩
])

ket_0 = np.array([1, 0])
ket_1 = np.array([0, 1])

# Clone |0⟩: CNOT|0⟩|0⟩ = |0⟩|0⟩ ✓
input_00 = np.kron(ket_0, ket_0)
output_00 = CNOT @ input_00
expected_00 = np.kron(ket_0, ket_0)
print("=== Attempting to Clone with CNOT ===")
print(f"CNOT|0⟩|0⟩ = {output_00} → |00⟩ ✓")

# Clone |1⟩: CNOT|1⟩|0⟩ = |1⟩|1⟩ ✓
input_10 = np.kron(ket_1, ket_0)
output_10 = CNOT @ input_10
expected_10 = np.kron(ket_1, ket_1)
print(f"CNOT|1⟩|0⟩ = {output_10} → |11⟩ ✓")

# Try to clone |+⟩: CNOT|+⟩|0⟩ = ?
ket_plus = (ket_0 + ket_1) / np.sqrt(2)
input_plus0 = np.kron(ket_plus, ket_0)
output_plus = CNOT @ input_plus0
expected_plus = np.kron(ket_plus, ket_plus)

print(f"\nCNOT|+⟩|0⟩ = {np.round(output_plus, 4)}")
print(f"|+⟩|+⟩     = {np.round(expected_plus, 4)}")
print(f"\nAre they equal? {np.allclose(output_plus, expected_plus)}")

print("\nCNOT|+⟩|0⟩ = (|00⟩+|11⟩)/√2 = |Φ⁺⟩ (a Bell state!)")
print("|+⟩|+⟩      = (|00⟩+|01⟩+|10⟩+|11⟩)/2 (a product state)")
print("\n→ CNOT entangles instead of cloning! No-cloning theorem confirmed.")

## 3. The Teleportation Protocol

### Setup

- **Alice** has an unknown state $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ that she wants to send to Bob
- Alice and Bob share an entangled Bell pair $|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$
- They have a classical communication channel

### The Three Qubits

```
Qubit 0 (q): Alice's unknown state |ψ⟩ = α|0⟩ + β|1⟩
Qubit 1 (a): Alice's half of the Bell pair
Qubit 2 (b): Bob's half of the Bell pair
```

### Protocol Steps

```
          q: |ψ⟩ ───────●───H───M──────────────────
                        │       ║
          a: |0⟩ ──H──●─⊕──────M─────────────────── 
                       │        ║  ║
          b: |0⟩ ──────⊕────────╫──╫───X^m1───Z^m0─── |ψ⟩
                                ║  ║
     classical:                 m0  m1
```

**Step 1:** Create Bell pair between qubits a and b  
**Step 2:** Alice applies CNOT(q, a) — entangles her unknown state with her Bell qubit  
**Step 3:** Alice applies H(q) — completes Bell measurement  
**Step 4:** Alice measures both her qubits (q and a), getting 2 classical bits (m0, m1)  
**Step 5:** Alice sends m0, m1 to Bob via classical channel  
**Step 6:** Bob applies corrections:  
- If m1 = 1: apply X to qubit b  
- If m0 = 1: apply Z to qubit b  

**Result:** Bob's qubit b is now in state $|\psi\rangle$!

## 4. Full Mathematical Derivation

### Initial State

The 3-qubit system starts in state:

$$|\Psi_0\rangle = |\psi\rangle_q \otimes |\Phi^+\rangle_{ab}$$

$$= (\alpha|0\rangle + \beta|1\rangle) \otimes \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$$

$$= \frac{1}{\sqrt{2}}[\alpha|0\rangle(|00\rangle + |11\rangle) + \beta|1\rangle(|00\rangle + |11\rangle)]$$

$$= \frac{1}{\sqrt{2}}[\alpha|000\rangle + \alpha|011\rangle + \beta|100\rangle + \beta|111\rangle]$$

### After CNOT(q, a)

CNOT flips qubit a (the target) when qubit q (control) is $|1\rangle$:

$$|\Psi_1\rangle = \frac{1}{\sqrt{2}}[\alpha|000\rangle + \alpha|011\rangle + \beta|110\rangle + \beta|101\rangle]$$

### After H(q)

Recall: $H|0\rangle = \frac{|0\rangle+|1\rangle}{\sqrt{2}}$, $H|1\rangle = \frac{|0\rangle-|1\rangle}{\sqrt{2}}$

$$|\Psi_2\rangle = \frac{1}{2}[(|0\rangle+|1\rangle)\alpha|00\rangle + (|0\rangle+|1\rangle)\alpha|11\rangle + (|0\rangle-|1\rangle)\beta|10\rangle + (|0\rangle-|1\rangle)\beta|01\rangle]$$

Expanding and grouping by the first two qubits (Alice's measurement outcomes):

$$|\Psi_2\rangle = \frac{1}{2}[|00\rangle(\alpha|0\rangle + \beta|1\rangle) + |01\rangle(\alpha|1\rangle + \beta|0\rangle) + |10\rangle(\alpha|0\rangle - \beta|1\rangle) + |11\rangle(\alpha|1\rangle - \beta|0\rangle)]$$

### The Magic Revealed!

$$|\Psi_2\rangle = \frac{1}{2}\bigg[|00\rangle \underbrace{(\alpha|0\rangle + \beta|1\rangle)}_{|\psi\rangle} + |01\rangle \underbrace{(\beta|0\rangle + \alpha|1\rangle)}_{X|\psi\rangle} + |10\rangle \underbrace{(\alpha|0\rangle - \beta|1\rangle)}_{Z|\psi\rangle} + |11\rangle \underbrace{(-\beta|0\rangle + \alpha|1\rangle)}_{XZ|\psi\rangle}\bigg]$$

### Measurement and Correction

| Alice measures | Bob's state | Bob's correction | After correction |
|:--------------:|:-----------:|:----------------:|:----------------:|
| $|00\rangle$ | $\alpha|0\rangle + \beta|1\rangle$ | None ($I$) | $|\psi\rangle$ |
| $|01\rangle$ | $\beta|0\rangle + \alpha|1\rangle$ | Apply $X$ | $|\psi\rangle$ |
| $|10\rangle$ | $\alpha|0\rangle - \beta|1\rangle$ | Apply $Z$ | $|\psi\rangle$ |
| $|11\rangle$ | $-\beta|0\rangle + \alpha|1\rangle$ | Apply $ZX$ | $|\psi\rangle$ |

Each outcome occurs with probability $1/4$, and in all cases Bob can recover $|\psi\rangle$!

In [ ]:
# Verify the derivation step by step
import numpy as np

# Define the initial state: |ψ⟩ = α|0⟩ + β|1⟩
alpha = np.sqrt(0.7) + 0j
beta = np.sqrt(0.3) * np.exp(1j * np.pi/3)  # Complex amplitude
psi = np.array([alpha, beta])

print(f"|ψ⟩ = {alpha:.4f}|0⟩ + {beta:.4f}|1⟩")
print(f"Norm: |α|² + |β|² = {abs(alpha)**2 + abs(beta)**2:.6f}")

# Basis states
ket_0 = np.array([1, 0], dtype=complex)
ket_1 = np.array([0, 1], dtype=complex)

# Bell state |Φ⁺⟩ = (|00⟩ + |11⟩)/√2
bell = (np.kron(ket_0, ket_0) + np.kron(ket_1, ket_1)) / np.sqrt(2)

# Initial 3-qubit state: |ψ⟩ ⊗ |Φ⁺⟩
psi_0 = np.kron(psi, bell)
print(f"\n|Ψ₀⟩ (8-dim): {np.round(psi_0, 4)}")

# Build CNOT on qubits 0,1 (identity on qubit 2)
CNOT_01 = np.kron(np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]]), np.eye(2))

# Build H on qubit 0 (identity on qubits 1,2)
H = np.array([[1,1],[1,-1]]) / np.sqrt(2)
H_0 = np.kron(np.kron(H, np.eye(2)), np.eye(2))

# Apply CNOT then H
psi_1 = CNOT_01 @ psi_0
psi_2 = H_0 @ psi_1

print(f"\n|Ψ₂⟩ after CNOT and H: {np.round(psi_2, 4)}")

# Extract Bob's state conditioned on Alice's measurement
print("\n=== Alice's Measurement → Bob's State ===")
corrections = {
    '00': np.eye(2),
    '01': np.array([[0,1],[1,0]]),  # X
    '10': np.array([[1,0],[0,-1]]), # Z
    '11': np.array([[0,1],[1,0]]) @ np.array([[1,0],[0,-1]]),  # XZ
}

for idx, (label, correction) in enumerate(corrections.items()):
    # Bob's state when Alice measures |label⟩
    bob_state = psi_2[2*idx:2*idx+2] * 2  # multiply by 2 to renormalize (prob = 1/4)
    corrected = correction @ bob_state
    
    print(f"\n  Alice measures |{label}⟩ (prob = 1/4):")
    print(f"    Bob's state: {np.round(bob_state, 4)}")
    print(f"    After correction: {np.round(corrected, 4)}")
    print(f"    Matches |ψ⟩? {np.allclose(corrected, psi) or np.allclose(corrected, -psi)}")

## 5. Step-by-Step Circuit Construction

Let's build the teleportation circuit piece by piece.

### Complete Circuit Diagram

```
         ┌──────────┐     ┌───┐┌───┐┌─┐
  q0: |ψ⟩─┤ Prepare ψ├─────┤ ● ├┤ H ├┤M├────────────────
         └──────────┘     └─│─┘└───┘└╥┘
                     ┌───┐  │        ║
  q1: |0⟩────────────┤ H ├──●───⊕────║──┤M├──────────────
                     └───┘  │        ║  ║
                            │        ║  ║  ┌───┐┌───┐
  q2: |0⟩───────────────────⊕────────╫──╫──┤ X ├┤ Z ├── |ψ⟩
                                     ║  ║  └─╥─┘└─╥─┘
                                     ║  ║    ║    ║
  c0: ═══════════════════════════════╩══╬════╬════╩════
                                        ║    ║
  c1: ══════════════════════════════════╩════╩═════════
```

### Phase 1: Bell Pair Creation (shared beforehand)
- H on qubit 1, CNOT(1, 2)

### Phase 2: Alice's Bell Measurement
- CNOT(0, 1), H on qubit 0
- Measure qubits 0 and 1

### Phase 3: Bob's Correction
- If classical bit 1 = 1: apply X to qubit 2
- If classical bit 0 = 1: apply Z to qubit 2

In [ ]:
# Build the teleportation circuit step by step
from qiskit import QuantumCircuit

def build_teleportation_circuit(prepare_state=None):
    """Build a quantum teleportation circuit.
    
    Qubits:
        q0: Alice's qubit (the state to teleport)
        q1: Alice's half of Bell pair
        q2: Bob's half of Bell pair (receives the state)
    """
    qc = QuantumCircuit(3, 2)  # 3 qubits, 2 classical bits
    
    # --- PHASE 0: Prepare the state to teleport ---
    if prepare_state is not None:
        prepare_state(qc)
    qc.barrier(label='Prep')
    
    # --- PHASE 1: Create Bell pair (q1, q2) ---
    qc.h(1)
    qc.cx(1, 2)
    qc.barrier(label='Bell')
    
    # --- PHASE 2: Alice's Bell measurement (q0, q1) ---
    qc.cx(0, 1)   # CNOT: control=q0, target=q1
    qc.h(0)        # Hadamard on q0
    qc.barrier(label='Meas')
    
    # Measure Alice's qubits
    qc.measure(0, 0)  # q0 → c0
    qc.measure(1, 1)  # q1 → c1
    qc.barrier(label='Corr')
    
    # --- PHASE 3: Bob's corrections ---
    qc.x(2).c_if(1, 1)   # If c1=1, apply X
    qc.z(2).c_if(0, 1)   # If c0=1, apply Z
    
    return qc

# Build with a simple state: teleport |1⟩
def prep_1(qc):
    qc.x(0)  # Prepare |1⟩

qc_teleport = build_teleportation_circuit(prep_1)
print("=== Quantum Teleportation Circuit ===")
print("(Teleporting |1⟩)")
print(qc_teleport.draw('text'))

## 6. Implementation with Qiskit

Let's implement and verify teleportation using Qiskit's simulator.

In [ ]:
# Teleportation with statevector simulation (no measurement needed)
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
import numpy as np

def teleport_statevector(alpha, beta):
    """Teleport state α|0⟩ + β|1⟩ using statevector simulation.
    Returns the final state of qubit 2 for each measurement outcome.
    """
    # Build circuit without measurement
    qc = QuantumCircuit(3)
    
    # Prepare state on q0
    qc.initialize([alpha, beta], 0)
    
    # Create Bell pair (q1, q2)
    qc.h(1)
    qc.cx(1, 2)
    
    # Bell measurement on (q0, q1)
    qc.cx(0, 1)
    qc.h(0)
    
    # Get statevector
    sv = Statevector.from_instruction(qc)
    state = sv.data
    
    # State is now: sum over |m0 m1⟩ |bob_state⟩
    # Extract Bob's state for each measurement outcome
    results = {}
    corrections = {
        '00': np.eye(2, dtype=complex),
        '01': np.array([[0,1],[1,0]], dtype=complex),
        '10': np.array([[1,0],[0,-1]], dtype=complex),
        '11': np.array([[0,-1],[-1,0]], dtype=complex),  # ZX
    }
    
    for idx, label in enumerate(['00', '01', '10', '11']):
        bob_state = state[2*idx:2*idx+2] * 2  # renormalize
        corrected = corrections[label] @ bob_state
        results[label] = corrected
    
    return results

# Teleport |ψ⟩ = √0.7|0⟩ + √0.3·e^(iπ/4)|1⟩
alpha = np.sqrt(0.7)
beta = np.sqrt(0.3) * np.exp(1j * np.pi/4)

print(f"=== Teleporting |ψ⟩ = {alpha:.4f}|0⟩ + {beta:.4f}|1⟩ ===")
print(f"P(0) = {abs(alpha)**2:.4f}, P(1) = {abs(beta)**2:.4f}")

results = teleport_statevector(alpha, beta)

for label, corrected in results.items():
    match = np.allclose(corrected, np.array([alpha, beta])) or np.allclose(corrected, -np.array([alpha, beta]))
    print(f"\n  Alice measures |{label}⟩:")
    print(f"    Bob's corrected state: [{corrected[0]:.4f}, {corrected[1]:.4f}]")
    print(f"    Matches |ψ⟩? {'YES ✓' if match else 'NO ✗'}")

In [ ]:
# Full teleportation with shot-based simulation
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import numpy as np

def teleportation_experiment(prep_fn, name, shots=4096):
    """Run teleportation and verify by measuring Bob's qubit."""
    simulator = AerSimulator()
    
    qc = QuantumCircuit(3, 3)  # 3 classical bits (including Bob's measurement)
    
    # Prepare state
    prep_fn(qc)
    qc.barrier()
    
    # Bell pair
    qc.h(1)
    qc.cx(1, 2)
    qc.barrier()
    
    # Bell measurement
    qc.cx(0, 1)
    qc.h(0)
    qc.measure(0, 0)
    qc.measure(1, 1)
    qc.barrier()
    
    # Corrections
    qc.x(2).c_if(1, 1)
    qc.z(2).c_if(0, 1)
    
    # Measure Bob's qubit
    qc.measure(2, 2)
    
    result = simulator.run(qc, shots=shots).result()
    counts = result.get_counts()
    
    # Analyze Bob's outcomes (bit 2, leftmost in Qiskit string)
    bob_0 = sum(v for k, v in counts.items() if k[0] == '0')
    bob_1 = sum(v for k, v in counts.items() if k[0] == '1')
    
    print(f"\n=== Teleporting {name} ===")
    print(f"  Bob measures |0⟩: {bob_0} ({bob_0/shots:.3f})")
    print(f"  Bob measures |1⟩: {bob_1} ({bob_1/shots:.3f})")
    return bob_0/shots, bob_1/shots

# Test 1: Teleport |0⟩
teleportation_experiment(lambda qc: None, "|0⟩ (expect: 100% |0⟩)")

# Test 2: Teleport |1⟩
teleportation_experiment(lambda qc: qc.x(0), "|1⟩ (expect: 100% |1⟩)")

# Test 3: Teleport |+⟩
teleportation_experiment(lambda qc: qc.h(0), "|+⟩ (expect: 50% each)")

# Test 4: Teleport custom state with P(0)=0.75
theta = 2 * np.arccos(np.sqrt(0.75))
teleportation_experiment(lambda qc: qc.ry(theta, 0), 
                        f"Ry({theta:.2f})|0⟩ (expect: P(0)=0.75)")

## 7. Verification: Teleporting Known States

Let's systematically verify teleportation works for the 6 cardinal states of the Bloch sphere.

In [ ]:
# Verify teleportation for all cardinal Bloch sphere states
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import numpy as np

simulator = AerSimulator()
shots = 10000

states_to_test = [
    ('|0⟩', lambda qc: None, {'0': 1.0, '1': 0.0}),
    ('|1⟩', lambda qc: qc.x(0), {'0': 0.0, '1': 1.0}),
    ('|+⟩', lambda qc: qc.h(0), {'0': 0.5, '1': 0.5}),
    ('|-⟩', lambda qc: [qc.x(0), qc.h(0)], {'0': 0.5, '1': 0.5}),
    ('|R⟩', lambda qc: [qc.h(0), qc.s(0)], {'0': 0.5, '1': 0.5}),
    ('|L⟩', lambda qc: [qc.h(0), qc.sdg(0)], {'0': 0.5, '1': 0.5}),
]

print("=== Teleportation Verification: All Cardinal States ===")
print(f"{'State':<8} {'Expected P(0)':<15} {'Observed P(0)':<15} {'Match?':<8}")
print("-" * 50)

for name, prep_fn, expected in states_to_test:
    qc = QuantumCircuit(3, 3)
    
    # Prepare
    prep_fn(qc)
    qc.barrier()
    
    # Bell pair
    qc.h(1)
    qc.cx(1, 2)
    qc.barrier()
    
    # Bell measurement
    qc.cx(0, 1)
    qc.h(0)
    qc.measure(0, 0)
    qc.measure(1, 1)
    qc.barrier()
    
    # Corrections
    qc.x(2).c_if(1, 1)
    qc.z(2).c_if(0, 1)
    
    # Measure Bob
    qc.measure(2, 2)
    
    result = simulator.run(qc, shots=shots).result()
    counts = result.get_counts()
    
    bob_0 = sum(v for k, v in counts.items() if k[0] == '0') / shots
    
    match = abs(bob_0 - expected['0']) < 0.05
    print(f"{name:<8} {expected['0']:<15.2f} {bob_0:<15.4f} {'✓' if match else '✗'}")

print("\n→ All states teleported successfully!")

## 8. Teleporting Arbitrary States

Let's teleport states at various points on the Bloch sphere and verify the fidelity.

In [ ]:
# Teleport states at many points on the Bloch sphere
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import numpy as np
import matplotlib.pyplot as plt

simulator = AerSimulator()
shots = 5000

# Vary theta from 0 to pi
thetas = np.linspace(0, np.pi, 20)
expected_p0 = np.cos(thetas/2)**2  # P(0) = cos²(θ/2)
observed_p0 = []

for theta in thetas:
    qc = QuantumCircuit(3, 3)
    
    # Prepare state: Ry(θ)|0⟩ = cos(θ/2)|0⟩ + sin(θ/2)|1⟩
    qc.ry(theta, 0)
    qc.barrier()
    
    # Bell pair
    qc.h(1)
    qc.cx(1, 2)
    qc.barrier()
    
    # Bell measurement
    qc.cx(0, 1)
    qc.h(0)
    qc.measure(0, 0)
    qc.measure(1, 1)
    qc.barrier()
    
    # Corrections
    qc.x(2).c_if(1, 1)
    qc.z(2).c_if(0, 1)
    
    # Measure Bob
    qc.measure(2, 2)
    
    result = simulator.run(qc, shots=shots).result()
    counts = result.get_counts()
    bob_0 = sum(v for k, v in counts.items() if k[0] == '0') / shots
    observed_p0.append(bob_0)

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(np.degrees(thetas), expected_p0, 'r-', linewidth=2, label='Expected P(0) = cos²(θ/2)')
ax.plot(np.degrees(thetas), observed_p0, 'bo', markersize=6, label='Teleported (observed)')
ax.set_xlabel('θ (degrees)', fontsize=13)
ax.set_ylabel('P(0) at Bob', fontsize=13)
ax.set_title('Quantum Teleportation: Ry(θ)|0⟩ teleported successfully', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 180])
ax.set_ylim([-0.05, 1.05])
plt.tight_layout()
plt.show()

# Compute average error
errors = np.abs(np.array(observed_p0) - expected_p0)
print(f"Average |error| = {np.mean(errors):.4f}")
print(f"Max |error| = {np.max(errors):.4f}")
print("→ Small errors due to finite sampling (statistical noise only, not protocol errors)")

## 9. Classical vs Quantum Communication

### What Teleportation IS and ISN'T

| Myth | Reality |
|------|--------|
| FTL communication | Classical bits must be sent (limited to speed of light) |
| Copies the state | Destroys the original (no-cloning) |
| Sends matter | Sends quantum information only |
| Needs no resources | Requires pre-shared entanglement + 2 classical bits |

### Resource Accounting

```
Teleportation requires:
  ┌──────────────────────────────────────────────────┐
  │  1 shared Bell pair (pre-distributed)            │
  │  + 2 classical bits (from Alice to Bob)          │
  │  ─────────────────────────────────────           │
  │  = 1 qubit of quantum information transferred    │
  │                                                  │
  │  The Bell pair is CONSUMED (used up)             │
  │  The original state is DESTROYED                 │
  └──────────────────────────────────────────────────┘
```

### Why Is Teleportation Useful?

1. **Quantum networks:** Transfer qubits between distant quantum computers
2. **Quantum error correction:** Move logical qubits between code blocks
3. **Gate teleportation:** Implement difficult gates using pre-prepared states
4. **Measurement-based QC:** The entire computation can be done via teleportation-like operations

### Comparison: Classical vs Quantum Channels

| Task | Classical | With Entanglement |
|------|-----------|-------------------|
| Send 1 classical bit | 1 cbit | 1 cbit (no advantage) |
| Send 1 qubit | Impossible (no-cloning) | 1 Bell pair + 2 cbits (teleportation) |
| Send 2 classical bits | 2 cbits | 1 qubit (superdense coding) |

In [ ]:
# Demonstrate that without corrections, teleportation fails
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import numpy as np

simulator = AerSimulator()
shots = 5000

# Prepare a state with known P(0) = 0.8
theta = 2 * np.arccos(np.sqrt(0.8))

# WITH corrections (correct teleportation)
qc_correct = QuantumCircuit(3, 3)
qc_correct.ry(theta, 0)
qc_correct.barrier()
qc_correct.h(1)
qc_correct.cx(1, 2)
qc_correct.barrier()
qc_correct.cx(0, 1)
qc_correct.h(0)
qc_correct.measure(0, 0)
qc_correct.measure(1, 1)
qc_correct.barrier()
qc_correct.x(2).c_if(1, 1)  # Correction
qc_correct.z(2).c_if(0, 1)  # Correction
qc_correct.measure(2, 2)

# WITHOUT corrections (broken teleportation)
qc_broken = QuantumCircuit(3, 3)
qc_broken.ry(theta, 0)
qc_broken.barrier()
qc_broken.h(1)
qc_broken.cx(1, 2)
qc_broken.barrier()
qc_broken.cx(0, 1)
qc_broken.h(0)
qc_broken.measure(0, 0)
qc_broken.measure(1, 1)
qc_broken.barrier()
# NO CORRECTIONS!
qc_broken.measure(2, 2)

# Run both
r_correct = simulator.run(qc_correct, shots=shots).result().get_counts()
r_broken = simulator.run(qc_broken, shots=shots).result().get_counts()

bob_0_correct = sum(v for k, v in r_correct.items() if k[0] == '0') / shots
bob_0_broken = sum(v for k, v in r_broken.items() if k[0] == '0') / shots

print("=== Importance of Classical Corrections ===")
print(f"\nOriginal state: Ry({theta:.2f})|0⟩ with P(0) = 0.80")
print(f"\n  WITH corrections:    Bob's P(0) = {bob_0_correct:.4f} (should be ~0.80) ✓")
print(f"  WITHOUT corrections: Bob's P(0) = {bob_0_broken:.4f} (should be ~0.50) ✗")
print("\n→ Without corrections, the teleported state is scrambled!")
print("  The 2 classical bits are ESSENTIAL for the protocol.")

In [ ]:
# Analyze the distribution of Alice's measurement outcomes
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import matplotlib.pyplot as plt

simulator = AerSimulator()
shots = 10000

# Teleport |+⟩
qc = QuantumCircuit(3, 3)
qc.h(0)  # Prepare |+⟩
qc.barrier()
qc.h(1)
qc.cx(1, 2)
qc.barrier()
qc.cx(0, 1)
qc.h(0)
qc.measure(0, 0)
qc.measure(1, 1)
qc.barrier()
qc.x(2).c_if(1, 1)
qc.z(2).c_if(0, 1)
qc.measure(2, 2)

result = simulator.run(qc, shots=shots).result()
counts = result.get_counts()

# Separate by Alice's outcomes
alice_outcomes = {'00': 0, '01': 0, '10': 0, '11': 0}
bob_given_alice = {'00': {'0': 0, '1': 0}, '01': {'0': 0, '1': 0},
                   '10': {'0': 0, '1': 0}, '11': {'0': 0, '1': 0}}

for bitstring, count in counts.items():
    bob_bit = bitstring[0]    # leftmost = qubit 2 (Bob)
    alice_bits = bitstring[1:]  # remaining = qubits 0,1 (Alice)
    alice_outcomes[alice_bits] += count
    bob_given_alice[alice_bits][bob_bit] += count

print("=== Teleporting |+⟩: Analysis by Alice's Measurement ===")
print(f"\n{'Alice measures':<16} {'Count':<8} {'Bob P(0)':<10} {'Bob P(1)':<10} {'Expected P(0)'}")
print("-" * 60)
for alice_bits in ['00', '01', '10', '11']:
    total = alice_outcomes[alice_bits]
    p0 = bob_given_alice[alice_bits]['0'] / total if total > 0 else 0
    p1 = bob_given_alice[alice_bits]['1'] / total if total > 0 else 0
    print(f"  |{alice_bits}⟩         {total:<8} {p0:<10.4f} {p1:<10.4f} 0.50")

print(f"\nTotal shots: {shots}")
print("\n→ Each Alice outcome occurs ~25% of the time (uniform distribution)")
print("→ Bob always gets ~50/50, regardless of Alice's outcome (after correction)")
print("→ The corrections ensure Bob's state matches |+⟩ for ALL outcomes")

In [ ]:
# Advanced: Teleportation using different Bell states
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import numpy as np

simulator = AerSimulator()
shots = 5000

# Instead of |Φ⁺⟩, use |Ψ⁺⟩ = (|01⟩+|10⟩)/√2 as the shared pair
# The corrections change accordingly!

print("=== Teleportation with |Ψ⁺⟩ Bell Pair ===")
print("Standard uses |Φ⁺⟩ = (|00⟩+|11⟩)/√2")
print("Now using |Ψ⁺⟩ = (|01⟩+|10⟩)/√2\n")

# Prepare state with P(0) = 0.9
theta = 2 * np.arccos(np.sqrt(0.9))

qc = QuantumCircuit(3, 3)
qc.ry(theta, 0)  # State to teleport
qc.barrier()

# Create |Ψ⁺⟩ instead of |Φ⁺⟩
qc.x(2)      # |0⟩ → |1⟩ on qubit 2
qc.h(1)
qc.cx(1, 2)  # Creates (|01⟩+|10⟩)/√2
qc.barrier()

# Bell measurement (same)
qc.cx(0, 1)
qc.h(0)
qc.measure(0, 0)
qc.measure(1, 1)
qc.barrier()

# Modified corrections for |Ψ⁺⟩
# With |Ψ⁺⟩, Bob's state has an extra X compared to |Φ⁺⟩
qc.x(2)               # Extra X correction for |Ψ⁺⟩
qc.x(2).c_if(1, 1)    # Standard correction
qc.z(2).c_if(0, 1)    # Standard correction
qc.measure(2, 2)

print(qc.draw('text'))

result = simulator.run(qc, shots=shots).result()
counts = result.get_counts()

bob_0 = sum(v for k, v in counts.items() if k[0] == '0') / shots
print(f"\nExpected P(0) = 0.90")
print(f"Observed P(0) = {bob_0:.4f}")
print(f"→ Teleportation works with any Bell state (with appropriate corrections)!")

## 10. Exercises

### Exercise 1: Teleport a 2-Qubit State
Extend the teleportation protocol to teleport a 2-qubit state $|\psi\rangle = \alpha|00\rangle + \beta|01\rangle + \gamma|10\rangle + \delta|11\rangle$. You will need 2 Bell pairs and 4 classical bits.

### Exercise 2: Superdense Coding
**Superdense coding** is the "reverse" of teleportation: Alice sends 2 classical bits using 1 qubit + shared entanglement. Implement it:
1. Share a Bell pair
2. Alice encodes 2 bits by applying {I, X, Z, XZ} to her qubit
3. Alice sends her qubit to Bob
4. Bob performs Bell measurement to decode

### Exercise 3: Fidelity Analysis
Add simulated noise to the teleportation circuit (using Qiskit Aer noise model) and measure how fidelity degrades with increasing noise.

### Exercise 4: Gate Teleportation
Research and implement **gate teleportation**: instead of teleporting a state, teleport a gate operation. This is key to fault-tolerant quantum computing.

### Exercise 5: Entanglement Swapping
If Alice-Bob share a Bell pair, and Bob-Charlie share a Bell pair, Bob can perform a Bell measurement to create entanglement between Alice and Charlie (who never interacted!). Implement this.

In [ ]:
# Exercise 2 Solution: Superdense Coding
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

simulator = AerSimulator()
shots = 1000

def superdense_coding(bits):
    """Send 2 classical bits using 1 qubit + shared entanglement.
    bits: string '00', '01', '10', or '11'
    """
    qc = QuantumCircuit(2, 2)
    
    # Step 1: Create shared Bell pair
    qc.h(0)
    qc.cx(0, 1)
    qc.barrier()
    
    # Step 2: Alice encodes 2 bits by operating on her qubit (q0)
    if bits == '00':   # Send 00: apply I (do nothing)
        pass
    elif bits == '01': # Send 01: apply X
        qc.x(0)
    elif bits == '10': # Send 10: apply Z
        qc.z(0)
    elif bits == '11': # Send 11: apply ZX
        qc.x(0)
        qc.z(0)
    qc.barrier()
    
    # Step 3: Alice sends her qubit to Bob
    # (In the circuit, this is implicit)
    
    # Step 4: Bob performs Bell measurement
    qc.cx(0, 1)
    qc.h(0)
    qc.measure([0, 1], [0, 1])
    
    return qc

print("=== Superdense Coding ===")
print("Alice sends 2 classical bits using 1 qubit + 1 Bell pair\n")

for bits in ['00', '01', '10', '11']:
    qc = superdense_coding(bits)
    result = simulator.run(qc, shots=shots).result()
    counts = result.get_counts()
    decoded = max(counts, key=counts.get)
    success = counts.get(bits, 0) / shots
    print(f"  Sent: {bits} → Decoded: {decoded} (success rate: {success:.1%})")

print("\n→ All 4 messages decoded with ~100% accuracy!")
print("  2 classical bits sent using only 1 qubit (+ pre-shared entanglement)")

In [ ]:
# Exercise 5 Solution: Entanglement Swapping
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

simulator = AerSimulator()
shots = 5000

print("=== Entanglement Swapping ===")
print("Alice-Bob share a Bell pair (q0, q1)")
print("Bob-Charlie share a Bell pair (q2, q3)")
print("Bob measures (q1, q2) → Alice & Charlie become entangled!\n")

qc = QuantumCircuit(4, 4)

# Bell pair 1: Alice (q0) - Bob (q1)
qc.h(0)
qc.cx(0, 1)
qc.barrier()

# Bell pair 2: Bob (q2) - Charlie (q3)
qc.h(2)
qc.cx(2, 3)
qc.barrier()

# Bob performs Bell measurement on his two qubits (q1, q2)
qc.cx(1, 2)
qc.h(1)
qc.measure(1, 1)
qc.measure(2, 2)
qc.barrier()

# Corrections on Charlie's qubit based on Bob's measurement
qc.x(3).c_if(2, 1)
qc.z(3).c_if(1, 1)

# Now measure Alice and Charlie
qc.measure(0, 0)
qc.measure(3, 3)

print(qc.draw('text'))

result = simulator.run(qc, shots=shots).result()
counts = result.get_counts()

# Check Alice-Charlie correlations (bits 0 and 3)
same = 0
diff = 0
for bitstring, count in counts.items():
    alice = bitstring[3]   # q0 (rightmost)
    charlie = bitstring[0]  # q3 (leftmost)
    if alice == charlie:
        same += count
    else:
        diff += count

print(f"\nAlice-Charlie correlation:")
print(f"  Same result: {same} ({same/shots:.1%})")
print(f"  Different:   {diff} ({diff/shots:.1%})")
print(f"\n→ Alice and Charlie are ~100% correlated!")
print("  They are now entangled, even though they NEVER interacted!")

## Summary

In this notebook, we covered:

1. **Quantum teleportation** — transferring a quantum state using entanglement + classical bits
2. **No-cloning theorem** — formal proof that arbitrary quantum states cannot be copied
3. **Teleportation protocol** — Bell pair creation, Bell measurement, classical correction
4. **Full mathematical derivation** — step-by-step expansion showing why it works
5. **Qiskit implementation** — both statevector and shot-based verification
6. **Verification** — teleported all cardinal Bloch sphere states and arbitrary states
7. **Classical vs quantum communication** — teleportation requires classical channel, no FTL
8. **Applications** — superdense coding, entanglement swapping

### Key Takeaways

1. Teleportation demonstrates the **power of entanglement** as a resource
2. The 2 classical bits are **essential** — without them, Bob's state is random
3. The original state is **destroyed** (no-cloning), making this a **move** not a **copy**
4. This protocol is foundational for **quantum networks**, **quantum error correction**, and **measurement-based quantum computing**

### Series Conclusion

This completes our 5-notebook series on quantum computing fundamentals:
1. Qubits and Gates
2. Superposition and Measurement
3. Quantum Entanglement
4. Quantum Circuits Deep Dive
5. Quantum Teleportation

You now have the foundational knowledge to explore quantum algorithms (Grover, Shor, VQE), quantum error correction, and quantum machine learning!